# Use Case 1: Sorting a Customer Support Ticket (the `Choice` primitive)

**Goal of this notebook:** learn the *one* simplest thing TypeSafe can do — pick the best option from a list of choices, given a piece of text.

We will pretend we run a helpdesk. A customer writes in, and we want the computer to automatically decide: is this ticket about **billing**, a **technical** problem, or **other**?

---

### What is TypeSafe, in plain words?

Normal AI chatbots write free-form paragraphs back to you. That's hard for a program to use reliably (you'd have to guess-parse the sentence).

TypeSafe is different: you ask it a **structured question** ("which of these 3 options fits?") and it gives you back a **structured answer** (the exact option, plus how confident it is). That makes it perfect for automation — sorting tickets, scoring things, answering yes/no questions — where your code needs a reliable, predictable answer, not a paragraph to interpret.

There are 3 basic question types (we'll cover all 3 in separate notebooks, one at a time):
- **Choice** ← we're doing this one today. "Pick one option out of a list."
- **Score** ← later. "Rate this on a scale."
- **Noul** ← later. "Is this true or false (yes/no)?"

Let's go step by step. Run each cell from top to bottom (click a cell, press `Shift+Enter`).

### 📬 Analogy

Imagine 3 mailboxes on a wall, labeled **Billing**, **Technical**, and **Other**. A person reads a customer's letter and has to drop it into exactly one mailbox.

- **Choice** = which mailbox they picked.
- **Probabilities** = how tempted they were by *each* mailbox before deciding.
- **Confidence** = how steady their hand was — shaky means it was a close call, steady means it was obvious.

**Example:** the letter says "I was charged twice, please refund the extra charge." The person reads "charged" and "refund", feels zero pull toward Technical or Other, and drops it straight into **Billing** with a steady hand. That becomes `choice = "billing"`, `confidence` near 1.0, and `probabilities = {billing: 1.0, technical: 0.0, other: 0.0}`.

Keep this picture in mind — everything below is just how we describe the mailboxes and the letter to the computer, and how we read back which mailbox it picked.

## Step 1 — Install the SDK

This downloads the small Python package that talks to TypeSafe's servers for us. You only need to run this once (it's saved on your computer afterwards).

In [ ]:
%pip install typesafe-sdk

## Step 2 — Add your API key

TypeSafe needs to know it's really *you* calling it, so it uses an API key (a secret password-like string), the same way most AI services do.

1. Get your key from the TypeSafe playground/console.
2. Paste it below, replacing the text `PASTE_YOUR_API_KEY_HERE` (keep the quote marks `"..."` around it).

⚠️ Treat this key like a password — don't share this notebook with the real key still typed into it.

In [ ]:
import os

# 👉 PASTE YOUR TYPESAFE API KEY BELOW, between the quotes
os.environ["TYPESAFE_API_KEY"] = "PASTE_YOUR_API_KEY_HERE"

print("Key saved for this session.")

## Step 3 — Create the client

The `TypeSafeClient` is just an object that knows how to send our questions to TypeSafe and bring back the answers. We create it once, and reuse it for every question in this notebook.

In [ ]:
from typesafe_sdk import Choice, TypeSafeClient

client = TypeSafeClient()
print("Client is ready.")

## Step 4 — The "state" (the text we want it to look at)

The **state** is just the input data — the thing TypeSafe should read before answering your question(s). Here, it's the text of a support ticket. Think of `state` as "the context / the document".

In [ ]:
ticket_text = "I was charged twice for my subscription this month. Please refund the extra charge ASAP."

state = {"document": ticket_text}

## Step 5 — Define the question

This is the important part. We create one `Choice` question named `"category"`:

- `instructions`: the actual question to ask, in plain English.
- `criteria`: a dictionary where each **key** is an option it's allowed to pick, and each **value** is a short description helping it understand that option.

TypeSafe will pick exactly one of the keys (`billing`, `technical`, or `other`) — never anything outside that list, and never a made-up sentence.

In [ ]:
questions = {
    "category": Choice(
        instructions="What is this support ticket about?",
        criteria={
            "billing": "Charges, invoices, payments, or refunds",
            "technical": "Something is broken, buggy, or not working",
            "other": "Anything that doesn't fit the categories above",
        },
    ),
}

## Step 6 — Send it to TypeSafe

`client.system_one(...)` is the one method that does all the work: it sends your `state` and `questions` to TypeSafe, and gets back the answers.

In [ ]:
response = client.system_one(
    state=state,
    questions=questions,
)

response

## Step 7 — Read the answer

The response has a `.choices` dictionary. We look up our question by the name we gave it (`"category"`). Each answer gives you three things:

- **`choice`** — the winning option (e.g. `"billing"`).
- **`confidence`** — a number from 0 to 1. Closer to 1 means "very sure", closer to 0 means "not very sure / it's a close call".
- **`probabilities`** — how likely *each* option was, so you can see what it considered and rejected.

In [ ]:
answer = response.choices["category"]

print("Chosen category :", answer.choice)
print("Confidence      :", answer.confidence)
print("All probabilities:", answer.probabilities)

You should see `billing` picked, with high confidence — because the ticket talks about being "charged twice" and wanting a "refund".

## Step 8 — Try it yourself

Same question, new ticket. Edit the `ticket_text_2` string below to whatever you like (e.g. a login bug, a spam complaint) and re-run this cell to see how the classification changes.

In [ ]:
# ✏️ Edit this text, then re-run this cell (Shift+Enter)
ticket_text_2 = "The app crashes every time I try to export a PDF."

response_2 = client.system_one(
    state={"document": ticket_text_2},
    questions=questions,  # reusing the same question we defined in Step 5
)

answer_2 = response_2.choices["category"]
print("Chosen category :", answer_2.choice)
print("Confidence      :", answer_2.confidence)
print("All probabilities:", answer_2.probabilities)

## Step 9 — Clean up

When we're done, we close the client's connection. (We didn't use the `with TypeSafeClient() as client:` shortcut from the docs, because that auto-closes at the end of a `with` block — not convenient across many separate notebook cells. Closing it manually here does the same job.)

In [ ]:
client.close()
print("Closed.")

## Recap — what did we just learn?

1. TypeSafe answers **structured questions** about text instead of writing free-form paragraphs.
2. `state` = the input text/data to look at.
3. `Choice(instructions=..., criteria={...})` = "pick exactly one of these labeled options."
4. `client.system_one(state=..., questions={...})` sends it off and returns a `response`.
5. `response.choices["your_question_name"]` gives you `.choice`, `.confidence`, and `.probabilities`.

That's the entire pattern for the `Choice` primitive — every classification use case (support tickets, routing, moderation categories, etc.) is built from this same shape.

👉 Once this makes sense to you, let's move on to the next notebook (the `Score` primitive — rating something on a scale).